# Wild-Diff-ICMH — Phase 3: H1 pooled pilot (λ = 2)

Một mô hình học từ **train của cả 20 site**, không phải một mô hình/site.
Resize cạnh dài 512, crop ngẫu nhiên 256; không dùng ROI loss hoặc H3.
**L4** là GPU đã chạy dự án. Chưa có kết quả Phase 3 trước khi bạn chạy notebook này.

Runtime mới: chạy **1 → 6**, rồi **7 (smoke một lần) → 8 → 9 → 10**.
Bước 8 đạt **500 optimizer steps tổng cộng**, không phải thêm 500 mỗi lần ấn.
Phiên hết thời gian: chạy lại Bước 8 để tiếp tục checkpoint, không tạo folder mới.
Bước 9 dùng 30 ảnh validation cố định, cùng protocol B0; không dùng test.
**11 và 12 tắt mặc định**: chỉ bật sau khi đã xem kết quả và ngân sách.

Khi có bản sửa: chạy **2A**, rồi đúng cell vừa lỗi. Pull cập nhật script/config
trong repo, không tự cập nhật các cell notebook đang mở. Nếu chính notebook
đổi cấu trúc, mở lại link này; không copy lẻ cell.
Code /content mất khi runtime bị thu hồi; checkpoint, log, kết quả trên Drive còn.
Không chạy lại notebook Phase 2, không tạo lại baseline/tags, không chạy Xie-SGC.

## 1 — Gắn Drive và ghi ngân sách phiên

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CU_AVAILABLE_AT_START = None  # điền số Available trong bảng Tài nguyên
CU_RATE = 1.54  # thay bằng tốc độ CU/giờ Colab đang báo; chỉ dùng để ước tính
SESSION_HOURS = 1.5  # giới hạn mỗi lần chạy training, có checkpoint cuối phiên


## 2 — Lấy code nhánh main / xem GPU (mỗi runtime mới)

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/TranDuon/Wild-Diff-ICMH.git'
BRANCH = 'main'
REPO = Path('/content/Wild-Diff-ICMH')

if (REPO / '.git').is_dir():
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)

os.chdir(REPO)
subprocess.run(['nvidia-smi'], check=True)
print('Mã nguồn:', REPO)


## 2A — Cell pull cố định: chạy sau khi được báo đã push bản sửa

In [ ]:
assert (REPO / '.git').is_dir(), 'Hãy chạy Bước 2 trước để clone repository.'

local_changes = subprocess.check_output(
    ['git', 'status', '--porcelain'], cwd=REPO, text=True
).strip()
if local_changes:
    raise RuntimeError(
        'Thư mục code Colab có sửa cục bộ nên chưa thể pull an toàn:\n'
        f'{local_changes}\n'
        'Không sửa file trong /content/Wild-Diff-ICMH; hãy gửi phần này để xử lý.'
    )

before = subprocess.check_output(
    ['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO, text=True
).strip()
subprocess.run(
    ['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO, check=True
)
after = subprocess.check_output(
    ['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO, text=True
).strip()
remote = subprocess.check_output(
    ['git', 'rev-parse', '--short', f'origin/{BRANCH}'], cwd=REPO, text=True
).strip()

print('Commit trước:', before)
print('Commit sau:  ', after)
print(f'GitHub {BRANCH}:', remote)
assert after == remote, f'Code local {after} chưa khớp GitHub {remote}'
if before == after:
    print('ĐÃ Ở BẢN MỚI NHẤT — chạy lại đúng cell vừa lỗi.')
else:
    print('CẬP NHẬT THÀNH CÔNG — chạy lại đúng cell vừa lỗi.')


## 3 — Cài môi trường đã kiểm tra cho Colab (giữ NumPy/SciPy/Torch)

In [ ]:
source_requirements = REPO / 'requirements-colab.txt'
safe_requirements = Path('/content/requirements-colab-safe.txt')
core_constraints = Path('/content/colab-core-constraints.txt')

from importlib import metadata as importlib_metadata

core_distributions = ('numpy', 'scipy', 'torch', 'torchvision')
core_versions_before = {
    name: importlib_metadata.version(name)
    for name in core_distributions
}

filtered_lines = []
for line in source_requirements.read_text(encoding='utf-8').splitlines():
    normalized = line.strip().lower()
    if normalized.startswith(('numpy', 'scipy')):
        print('Giữ bản Colab, bỏ qua:', line)
        continue
    filtered_lines.append(line)

safe_requirements.write_text(
    '\n'.join(filtered_lines) + '\n',
    encoding='utf-8',
)

core_constraints.write_text(
    '\n'.join(
        f'{name}=={version}'
        for name, version in core_versions_before.items()
    ) + '\n',
    encoding='utf-8',
)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(safe_requirements),
    '-c', str(core_constraints),
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--no-deps', '--no-build-isolation',
    'compressai==1.2.8',
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    '--force-reinstall', 'src/recognize-anything',
], cwd=REPO, check=True)

import importlib
import numpy as np
import numpy.random as npr
import scipy
import torch
import torch_geometric
import pytorch_msssim
import compressai.ans
import compressai.entropy_models
import compressai.losses
import compressai.zoo

# A pip subprocess can add files after this Jupyter kernel initialized.
# Refresh import caches so the freshly installed vendored package is
# visible immediately, without requiring a runtime restart.
importlib.invalidate_caches()
import ram
from model.lfgcm import TagGCM

core_versions_after = {
    name: importlib_metadata.version(name)
    for name in core_distributions
}
assert core_versions_after == core_versions_before, (
    'Dependency install changed Colab core packages: '
    f'before={core_versions_before}, after={core_versions_after}'
)
assert torch.cuda.is_available(), 'Vào Runtime > Change runtime type > chọn GPU'
gpu = torch.cuda.get_device_properties(0)
print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__, np.__file__)
print('SciPy:', scipy.__version__)
print('Torch:', torch.__version__, torch.version.cuda)
print('PyG:', torch_geometric.__version__)
print('RAM package:', ram.__file__)
print('Core packages preserved:', core_versions_after)
print('GPU:', gpu.name, f'{gpu.total_memory / 2**30:.1f} GiB')
print('Kiểm tra numpy.random:', npr.rand(3))
print('THÀNH CÔNG: môi trường và TagGCM đã sẵn sàng.')


## 4 — Chép ảnh đã có trên Drive, kiểm tra split (không tải lại dataset)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

DRIVE_ROOT = Path('/content/drive/MyDrive/wild_diff_icmh')
DRIVE_IMAGES = DRIVE_ROOT / 'images' / 'snapshot_kgalagadi'
LOCAL_IMAGES = Path('/content/data/wild_diff_icmh/images')
LOCAL_SITE_IMAGES = LOCAL_IMAGES / 'snapshot_kgalagadi'

assert DRIVE_IMAGES.is_dir(), f'Không thấy dữ liệu: {DRIVE_IMAGES}'
LOCAL_SITE_IMAGES.mkdir(parents=True, exist_ok=True)

print('Đang quét file trên Drive...')
source_files = [path for path in DRIVE_IMAGES.rglob('*') if path.is_file()]
pending = []
for src in source_files:
    dst = LOCAL_SITE_IMAGES / src.relative_to(DRIVE_IMAGES)
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        pending.append((src, dst, src.stat().st_size))

remaining_bytes = sum(size for _, _, size in pending)
print('Tổng file:', len(source_files))
print('Còn phải copy:', len(pending), 'file')
print('Dung lượng còn lại:', f'{remaining_bytes / 2**30:.2f} GiB')

def copy_one(item):
    src, dst, size = item
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.part')
    try:
        with src.open('rb') as fin, tmp.open('wb') as fout:
            while chunk := fin.read(8 * 1024 * 1024):
                fout.write(chunk)
        os.replace(tmp, dst)
        return size
    except Exception:
        if tmp.exists():
            tmp.unlink()
        raise

if pending:
    with ThreadPoolExecutor(max_workers=6) as executor:
        futures = [executor.submit(copy_one, item) for item in pending]
        with tqdm(
            total=remaining_bytes,
            unit='B', unit_scale=True, unit_divisor=1024,
            desc='Copy Kgalagadi',
        ) as progress:
            for future in as_completed(futures):
                progress.update(future.result())
else:
    print('Ảnh cục bộ đã đầy đủ, không cần copy lại.')

subprocess.run([
    sys.executable, 'tools/data/split_check.py',
    'data/manifests/kgalagadi_site_split.jsonl',
    '--build-info', 'data/manifests/build_info.json',
], cwd=REPO, check=True)
print('Ảnh và manifest đã sẵn sàng.')


## 5 — Chép checkpoint gốc λ = 2 và Stable Diffusion (không tải lại)

In [ ]:
DRIVE_CKPT_ROOT = DRIVE_ROOT / 'checkpoints'
CKPT_ROOT = Path('/content/data/wild_diff_icmh/checkpoints')
BPP_WEIGHT = 2
folder = f'CNscale1.0_1_1_{BPP_WEIGHT}_2_WTagGCM_bs16x1_lr0.00005_cfg7.0'

checkpoint_relatives = [
    Path('sd2p1/v2-1_512-ema-pruned.ckpt'),
    Path('ram/ram_plus_swin_large_14m.pth'),
    Path('difficmh_models') / folder / 'model.ckpt',
]
backup_items = []
for relative in checkpoint_relatives:
    src = DRIVE_CKPT_ROOT / relative
    dst = CKPT_ROOT / relative
    assert src.is_file(), f'Thiếu checkpoint trên Drive: {src}'
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        backup_items.append((src, dst, src.stat().st_size))

total_bytes = sum(size for _, _, size in backup_items)
if backup_items:
    def copy_checkpoint(item, progress):
        src, dst, _ = item
        dst.parent.mkdir(parents=True, exist_ok=True)
        tmp = dst.with_name(dst.name + '.part')
        try:
            with src.open('rb') as fin, tmp.open('wb') as fout:
                while chunk := fin.read(8 * 1024 * 1024):
                    fout.write(chunk)
                    progress.update(len(chunk))
            os.replace(tmp, dst)
            return dst
        except Exception:
            if tmp.exists():
                tmp.unlink()
            raise

    with tqdm(
        total=total_bytes,
        unit='B', unit_scale=True, unit_divisor=1024,
        desc='Khôi phục checkpoint',
    ) as progress:
        with ThreadPoolExecutor(max_workers=2) as executor:
            futures = [
                executor.submit(copy_checkpoint, item, progress)
                for item in backup_items
            ]
            for future in as_completed(futures):
                print('\nĐã khôi phục:', future.result())
else:
    print('Checkpoint cục bộ đã đầy đủ, không cần copy lại.')

AUTHOR_CKPT = CKPT_ROOT / 'difficmh_models' / folder / 'model.ckpt'
RAM_CKPT = CKPT_ROOT / 'ram' / 'ram_plus_swin_large_14m.pth'
repo_checkpoints = REPO / 'checkpoints'
if not repo_checkpoints.exists():
    repo_checkpoints.symlink_to(CKPT_ROOT, target_is_directory=True)
elif repo_checkpoints.resolve() != CKPT_ROOT.resolve():
    raise RuntimeError(f'{repo_checkpoints} không trỏ tới {CKPT_ROOT}')

print('RAM++:', RAM_CKPT)
print('Diff-ICMH:', AUTHOR_CKPT)
print('Checkpoint đã sẵn sàng.')


## 6 — Kiểm tra tài sản Phase 2 / đóng băng 30 ảnh validation (CPU)

Không tạo RAM tags mới. Cần KGA_all.jsonl, detections/originals.jsonl,
kgalagadi_illumination.jsonl và kgalagadi_dev_b0eval.txt trên Drive.
Tập 30 được phân tầng ngày/đêm, ảnh trống và kích thước con vật; đây là
tập kiểm tra chọn mô hình, không đại diện tỷ lệ tự nhiên của toàn corpus.

In [ ]:
import json
import shutil

def p3(action, *options):
    # Each call launches the currently pulled helper, not cached imports.
    subprocess.run([
        sys.executable, '-u', 'tools/phase3_h1.py', action,
        '--drive-root', str(DRIVE_ROOT), '--cu-rate', str(CU_RATE), *map(str, options),
    ], cwd=REPO, check=True)

p3('prepare')
print('Run H1:', DRIVE_ROOT / 'runs/h1_pooled_pilot_ls512/lambda_2')


## 7 — Smoke cấu hình mới: train 20 step, decode 4 ảnh × 5 bước (một lần)

Run smoke riêng, không dùng checkpoint smoke làm kết quả pilot.
Chạy lại tự bỏ qua train đã đủ / ảnh đã decode. Không so DDIM5 với B0 DDIM50.

In [ ]:
p3('train', '--target-steps', 20, '--session-hours', SESSION_HOURS)
p3('evaluate', '--scope', 'smoke')


## 8 — Train pilot đến 500 optimizer steps (GPU, việc chính)

Không sinh ảnh trong training; validation chỉ tính loss trên 2 batch.
Checkpoint rolling mỗi 100 optimizer steps, lưu đầy đủ trạng thái cuối phiên.
Nếu runtime bị ngắt đột ngột, có thể mất phần sau checkpoint gần nhất.
Giới hạn mỗi phiên SESSION_HOURS; chưa đủ 500 thì chạy lại cell này.

In [ ]:
p3('train', '--target-steps', 500, '--session-hours', SESSION_HOURS)
p3('status')


## 9 — Đánh giá nhanh 30 ảnh và đối chiếu B0 đã lưu

Snapshot checkpoint cố định trước khi decode. Cả B0 và H1: cạnh dài 512,
DDIM50, seed231, guidance3, ảnh tái tạo đánh giá ở kích thước gốc.
Đo dung lượng, chất lượng ảnh (gồm LPIPS), giữ con vật/mất con vật/phát hiện
giả bằng MegaDetector. Không chạy SpeciesNet/DISTS tại pilot.
MegaDetector gốc là nhãn tự động, không phải hộp chuẩn do người gán.
Có ảnh/kết quả đủ rồi thì bỏ qua; chỉ chấm lại B0 trên tập nhỏ, không decode B0.

In [ ]:
def run_logged(command, log_name, echo=True):
    log_path = DRIVE_ROOT / 'logs' / log_name
    command = [str(part) for part in command]
    print('$', ' '.join(command))
    with log_path.open('a', encoding='utf-8') as log_stream:
        process = subprocess.Popen(
            command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in process.stdout:
            if echo:
                print(line, end='')
            log_stream.write(line)
        return_code = process.wait()
    if return_code:
        raise RuntimeError(f'{log_name}: dừng với mã {return_code}; log đầy đủ: {log_path}')

# Colab's Python has no ensurepip, so ``python -m venv`` cannot install pip;
# virtualenv ships its own.  --system-site-packages reuses Colab's torch.
DETECT_ENV = Path('/content/envs/detect')
DETECT_PY = DETECT_ENV / 'bin' / 'python'
DETECT_READY = DETECT_ENV / '.ready'
if not DETECT_READY.exists():
    shutil.rmtree(DETECT_ENV, ignore_errors=True)  # leftovers of a failed attempt
    run_logged([sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv'], 'p2_detect_env.log')
    run_logged([sys.executable, '-m', 'virtualenv', '--system-site-packages', DETECT_ENV], 'p2_detect_env.log')
    # The env also sees Colab's transformers, which needs Colab's
    # huggingface-hub (<1.0); PytorchWildlife's deps would otherwise pull 1.x.
    import importlib.metadata
    hub_pin = f"huggingface-hub=={importlib.metadata.version('huggingface_hub')}"
    run_logged([DETECT_PY, '-m', 'pip', 'install', '-q', 'PytorchWildlife', hub_pin], 'p2_detect_env.log')
    run_logged([DETECT_PY, '-c', 'from PytorchWildlife.models import detection; print("PytorchWildlife OK")'],
               'p2_detect_env.log')
    DETECT_READY.touch()

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pycocotools'], check=True)
p3('evaluate', '--scope', 'quick', '--detect-python', DETECT_PY)


## 10 — Xem chi phí / gửi kết quả để quyết định bước tiếp theo

In [ ]:
p3('status')
comparisons = sorted((DRIVE_ROOT / 'phase3/eval').glob('step_*_quick/comparison.json'))
for path in comparisons:
    report = json.loads(path.read_text())
    print('Kết quả:', path)
    for method, values in report['methods'].items():
        quality = values['quality']['illumination=all|content=all']
        machine = values['machine']['illumination=all|content=all']
        print(method, {key: quality.get(key) for key in ('n', 'bpp', 'psnr', 'lpips')})
        print(method, {key: machine.get(key) for key in ('map', 'missed_animal_rate', 'hallucination_rate')})
print('Gửi comparison.json + throughput.json. Không tự kết luận H1 tốt hơn từ 30 ảnh.')


## 11 — Tùy chọn: kéo dài đến 1.000 hoặc 3.000 step (TẮT)

Chỉ bật khi đã review pilot, đủ ngân sách và được chọn tiếp tục.
Đây là tổng step, resume đúng run; không train lại từ đầu. Sau đó chạy 9 → 10.

In [ ]:
RUN_EXTENSION = False
EXTEND_TO = 1000  # chỉ 1000 hoặc 3000
if RUN_EXTENSION:
    assert EXTEND_TO in (1000, 3000)
    p3('train', '--target-steps', EXTEND_TO, '--session-hours', SESSION_HOURS, '--allow-extension')
else:
    print('Bỏ qua kéo dài training.')


## 12 — Tùy chọn: đánh giá toàn bộ dev202 của mô hình được chọn (TẮT)

Chỉ bật sau khi chọn checkpoint cuối. Không chạy lại baseline, không dùng test.
Tốn hơn dev30; xem ngân sách trước. Không tự bật qua Run all.

In [ ]:
RUN_FULL_DEV = False
if RUN_FULL_DEV:
    p3('evaluate', '--scope', 'full', '--allow-full', '--detect-python', DETECT_PY)
else:
    print('Bỏ qua full dev; Phase 3 chưa được confirm hoàn thành.')


## 13 — Kết thúc phiên: ghi CU thực và ngắt GPU (chủ động bật)

In [ ]:
CU_AVAILABLE_AT_END = None  # ghi số Available mới
if CU_AVAILABLE_AT_START is not None and CU_AVAILABLE_AT_END is not None:
    print('CU thực tiêu cả phiên:', CU_AVAILABLE_AT_START - CU_AVAILABLE_AT_END)
DISCONNECT_NOW = False  # bật khi các việc cần chạy đã xong để không giữ GPU nhàn rỗi
if DISCONNECT_NOW:
    from google.colab import runtime
    drive.flush_and_unmount()
    runtime.unassign()
